##Prerequisties

####Load users_001.csv into dataframe

In [0]:
df = spark.read.csv('/Volumes/wns24082026/quickstart_schema/sandbox/datasets/user_dataset/users_001.csv', header=True, inferSchema=True)

###Transaction 01 - Write data in form of delta

In [0]:
df.write.format("DElTA").save("/Volumes/wns24082026/quickstart_schema/sandbox/output_delta")

###read delta

In [0]:
spark.read.load("/Volumes/wns24082026/quickstart_schema/sandbox/output_delta").display()


###Transaction 2 - Overwrite Delta

In [0]:
from pyspark.sql.functions import *

df.filter(col("country")=='India').write.save("/Volumes/wns24082026/quickstart_schema/sandbox/output_delta", mode="overwrite")

In [0]:
spark.read.load("/Volumes/wns24082026/quickstart_schema/sandbox/output_delta").display()

###Advantage - Maintain Versions

In [0]:
spark.read.option("versionAsOf", 0).load("/Volumes/wns24082026/quickstart_schema/sandbox/output_delta").display()

###Read Transaction log

In [0]:
spark.read.load(format="text",path="/Volumes/wns24082026/quickstart_schema/sandbox/output_delta/_delta_log/00000000000000000000.json").show(truncate=False)

###Approach 02

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/Volumes/wns24082026/quickstart_schema/sandbox/output_delta")

delta_table.history().display()

   1. Read Delta as Dataframe
   2. Create a view on df
   3. Updates are allowed

In [0]:
df = spark.read.load("/Volumes/wns24082026/quickstart_schema/sandbox/output_delta")
df.createOrReplaceTempView("users_vw")

In [0]:
%sql

UPDATE users_vw
SET country = 'Bharat'
WHERE country = 'India'

In [0]:
%sql

Select * from users_vw

In [0]:
df.show()